# 02c Estimate: the regression models with indicators

**Purpose:** The second model family: the three regressions of `02b_estimate_regression.ipynb` (linear regression, Ridge, Lasso) with indicators next to the series' own past, in the same backtest and with the same rules. **Part A** (sections 1 to 7, before the test): the route indicator of our plan, one block. **Part B** (sections B1 to B8, added after the test): every indicator of `01b` alone, all of them together ("all indicators"), and all of them minus one block at a time, so that `03_evaluate.ipynb` can show which data lower the error the most.
**Input:** `../data/processed/asylum_clean.csv` (from `01_preprocess.ipynb`) and the files of `01b_preprocess_indicators.ipynb` (`routes_clean.csv`; for Part B also `extra_indicators_clean.csv`, `nationality_shares_clean.csv`, `conflict_intensity_clean.csv`).
**Output:** `../data/processed/backtest_forecasts_indicator.csv` (Part A); `../data/processed/backtest_forecasts_extra.csv` and `../output/tables/02c_lasso_kept.csv` (Part B).

As before, the test period (2024-10 to 2026-09) is **not used**. We compare the models in `03_evaluate.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## 1. Load the target and the indicator

We put the log of the indicator next to the target, month by month. The indicator starts in January 2009, the target in January 2008, so the first 12 months of the indicator are missing (`NaN`). `reindex` puts the indicator on the months of the target.

*Not seen in class:* `reindex(months)` puts a series on a given list of row labels and leaves `NaN` where a label is missing.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")

routes = pd.read_csv("../data/processed/routes_clean.csv")
routes["month"] = pd.to_datetime(routes["month"])
routes = routes.set_index("month")

y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
months = clean.index
log_routes = routes["log_routes"].reindex(months).to_numpy(dtype=float)   # NaN before 2009-01

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])

print("Indicator missing for", np.isnan(log_routes).sum(), "months (before", routes.index[0], ")")

Indicator missing for 12 months (before 2009-01-01 00:00:00 )


## 2. The two indicator features

Step C fixed the rule: **at a forecast origin $t$, the newest indicator month we may use is $t-1$** (Frontex is published one month later than the target). We decided on two features before running any model, with $R$ = detections on the three routes:

| Feature | Formula | Idea |
|---|---|---|
| `routes_change_last_month` | $\log R_{t-1} - \log R_{t-2}$ | the newest monthly change on the routes |
| `routes_change_3_months` | $\log R_{t-1} - \log R_{t-4}$ | the change over the last three known months |

They are **changes**, like the other features, for the reason of notebook 200 (trending levels can look related without any link). We did not use a change over a whole year: it would need the indicator 13 months back, and the first training windows would become very small (in the first validation block, only about 6 rows for 16 features).

All other features, the target, the models, the grids of `alpha` and the validation inside each training window are **exactly those of `02b_estimate_regression.ipynb`**.

In [3]:
def make_table(h):
    # Same table as in 02b, plus the two indicator features (indicator months up to t - 1 only)
    rows = []
    for t in range(12, len(log_y) - h):
        if np.isnan(log_routes[t - 4]):           # the indicator is needed back to month t - 4
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            "routes_change_last_month": log_routes[t - 1] - log_routes[t - 2],
            "routes_change_3_months": log_routes[t - 1] - log_routes[t - 4],
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    features = ["change_last_month", "change_month_before", "same_change_last_year",
                "routes_change_last_month", "routes_change_3_months"]
    X = pd.concat([table[features], dummies], axis=1)
    return X, table["target"]

tables = {h: make_table(h) for h in range(1, H_MAX + 1)}

X2, target2 = tables[2]
print("Table for h = 2:", X2.shape[0], "rows and", X2.shape[1], "features")
print("First row: origin", months[X2.index.min()], "| missing values:", X2.isna().sum().sum())
X2[["routes_change_last_month", "routes_change_3_months"]].describe().round(3)

Table for h = 2: 207 rows and 16 features
First row: origin 2009-05-01 00:00:00 | missing values: 0


,routes_change_last_month,routes_change_3_months
count,207.000,207.000
mean,0.002,0.006
std,0.376,0.716
min,-1.645,-2.216
25%,-0.229,-0.475
50%,0.039,0.093
75%,0.218,0.479
max,1.227,2.384


**What we see.** For $h = 2$, the table has 207 rows and 16 features: the three own changes, the two route changes and 11 month dummies. No value is missing. The first row is the origin May 2009: the 3-month route change needs the indicator back to $t - 4$, and the indicator starts in January 2009. (In `02b` the table starts in January 2009, so the indicator models have 4 fewer training rows.)

The two route features are centred near 0 (means 0.002 and 0.006) but they move a lot: the standard deviation is 0.38 for the monthly change and 0.72 for the 3-month change, and the largest monthly fall is −1.65 in logs. The route numbers jump much more from month to month than the applications in Switzerland.

## 3. Models, grids and the choice of `alpha`: the same as in `02b`

The functions below are copied from `02b_estimate_regression.ipynb` without any change (scaling inside the pipeline, two validation blocks of 12 months inside each training window, the same grids).

In [4]:
def make_model(name, alpha=None):
    if name == "Linear regression + routes":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge + routes":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression + routes": [None],
    "Ridge + routes": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso + routes": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def choose_alpha(name, X, target, T, h, n_blocks=2, block=12):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    first_row, last_row = X.index.min(), T - h
    scores = []
    for alpha in grid:
        mse_blocks = []
        for b in range(n_blocks):
            start = last_row - n_blocks * block + 1 + b * block
            valid_rows = list(range(start, start + block))
            train_rows = list(range(first_row, start - h + 1))
            model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
            mse_blocks.append(np.mean(errors ** 2))
        scores.append(np.mean(mse_blocks))
    return grid[int(np.argmin(scores))]

## 4. The backtest loop

The same loop as in `02b`: the same 141 origins and the same horizons. We check again that no training target lies after the origin (`assert`).

In [5]:
rows = []

for t in range(first_origin, last_backtest):
    for h in range(1, H_MAX + 1):
        target_pos = t + h
        if target_pos > last_backtest:
            continue
        X, target = tables[h]
        train_rows = list(range(X.index.min(), t - h + 1))
        assert max(train_rows) + h <= t                    # no training target after the origin

        for name in alpha_grids:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                alpha = choose_alpha(name, X, target, t, h)
                model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
            rows.append({
                "model": name, "origin": months[t], "month": months[target_pos], "h": h,
                "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha,
            })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 1260


## 5. Checks on the stored forecasts

In [6]:
print(forecasts.groupby(["model", "h"]).size())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

model                       h
Lasso + routes              1    141
                            2    140
                            3    139
Linear regression + routes  1    141
                            2    140
                            3    139
Ridge + routes              1    141
                            2    140
                            3    139
dtype: int64

All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True
Missing or infinite forecasts: 0


**What we see.** As in `02b`: 141, 140 and 139 forecasts per model for $h = 1, 2, 3$ (1,260 together), all in the backtest, all after their origin, none missing. Together with the `assert` in the loop, this shows that the indicator models used only data known at each origin and never touched the test period.

### Which penalties were chosen?

As in `02b` (notebook 209), we check that the chosen `alpha` does not sit mostly at an edge of the grid.

In [7]:
for name in ["Ridge + routes", "Lasso + routes"]:
    chosen = forecasts.loc[forecasts["model"] == name, "alpha"]
    print(f"{name}: how often each alpha was chosen (grid from {alpha_grids[name][0]} to {alpha_grids[name][-1]})")
    print(chosen.value_counts().sort_index().to_string())
    print()

Ridge + routes: how often each alpha was chosen (grid from 0.1 to 10000)
alpha
0.1         92
1.0          9
10.0        50
100.0      196
1000.0      65
10000.0      8

Lasso + routes: how often each alpha was chosen (grid from 0.0001 to 0.3)
alpha
0.0001    100
0.0010     11
0.0030     22
0.0100     93
0.0300     78
0.1000     66
0.3000     50



**What we see.**
- **Ridge + routes** most often chose `alpha = 100` (196 of 420 times). The largest value (10,000) was chosen 8 times, the smallest (0.1) 92 times.
- **Lasso + routes** most often chose 0.0001 (100 times), then 0.01 (93) and 0.03 (78). The largest value (0.3) was chosen 50 times.

**What this means for us.** The upper edges are rarely reached. The lower edge means almost no penalty, which is the linear regression we already run. This is the same pattern as in `02b`, so we keep the same grids. Keeping them also keeps the comparison fair: only the indicator block changes.

## 6. What do the models do with the indicator?

As in `02b`, we look at the coefficients for $h = 2$ at the last origin (August 2024), on standardised features (notebook 105). We show only the five non-calendar features. Again: a coefficient says what helps to *predict*, not what *causes*.

In [8]:
t = last_backtest - 1
h = 2
X, target = tables[h]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in ["Ridge + routes", "Lasso + routes"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_     # the last step of the pipeline is the model

print("Origin:", months[t], "| horizon h =", h, "| training rows:", len(train_rows))
coefficients.iloc[:5].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | training rows: 182


,Ridge + routes (alpha = 0.1),Lasso + routes (alpha = 0.0001)
change_last_month,-0.016,-0.015
change_month_before,-0.016,-0.016
same_change_last_year,-0.001,-0.001
routes_change_last_month,0.021,0.021
routes_change_3_months,-0.053,-0.053


**What we see.** At the last origin (182 training rows), Ridge and Lasso chose their smallest penalty and give almost the same coefficients. Among these five features, the 3-month route change has the largest coefficient in size (−0.053), and the monthly route change has +0.021. The two own recent changes have about −0.016 each, and the change one year earlier −0.001.

**What this means for us.** The models do use the indicator: its coefficients are not zero. But the two route features have opposite signs, and the 3-month change contains the last monthly change (both use $\log R_{t-1}$). So the two partly cancel, and we should not read each sign on its own. The real question is whether the indicator makes the *forecasts* better, which we check in `03_evaluate.ipynb`.

## 7. Save the forecasts

In [9]:
forecasts.to_csv("../data/processed/backtest_forecasts_indicator.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 1260 forecasts with the columns ['model', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha']


# Part B: all indicators (added after the test)

Part A added one block, the Frontex routes, and it did not help (`03_evaluate.ipynb`, section 7). After the test we added the other indicators of `01b` (Part B). Each block enters with two features, like the routes in Part A, always ending at the newest month known at the cutoff. We run:

- each block **alone**;
- **all indicators** together: neighbours, Mediterranean routes and conflict index (6 extra features), the model with the most data;
- **all indicators but one** block, three times: if the error rises when a block is removed, that block was helping.

**Why the Frontex routes of Part A are not in this family.** The Mediterranean routes contain the same numbers for the Central and Eastern Mediterranean routes (correlations 0.99 and 0.96, `01b`, section B3), but they are known **one month earlier** (UNHCR timing $t$ instead of Frontex $t-1$). Both route series lead Swiss applications by one month only (`01b`, sections 5 and B8), so only the earlier timing can use the lead. In the backtest, the routes alone made Ridge 1.7% worse while the Mediterranean routes alone made it 4.8% better, and in a first run with both blocks in the model, removing the Mediterranean routes raised the MSE (log) by 5% to 10%, removing the Frontex routes by at most 2.7%. So we keep one route block, the Mediterranean routes at the UNHCR timing. We lose the Western Balkan land route; part of its signal is in Austria's applications, which are in the neighbours block. The decision uses the backtest only. The Frontex routes stay in Part A as the indicator of our plan.

The models, grids, validation blocks and backtest origins are those of `02b` and Part A. Everything else is copied from Part A or `02b`, so any change in the errors comes from the data.

In [10]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## B1. Load the target and the extra data

As in Part A, `reindex` puts every series on the months of the target.

In [11]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
months = clean.index
y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

extra = load("../data/processed/extra_indicators_clean.csv").reindex(months)
shares = load("../data/processed/nationality_shares_clean.csv").reindex(months)
intensity = load("../data/processed/conflict_intensity_clean.csv")

log_neighbours = extra["log_neighbours"].to_numpy(dtype=float)
log_med = extra["log_med_routes"].to_numpy(dtype=float)

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])
print("Neighbours missing:", np.isnan(log_neighbours).sum(), "| Med routes missing:", np.isnan(log_med).sum())

Neighbours missing: 1 | Med routes missing: 12


## B2. The conflict index

For a forecast origin $t$, with $w_{c,t}$ the share of nationality $c$ in the Swiss applications of the 12 months up to $t$ (top 10 nationalities only) and $I_{c,s} = \log(1 + \text{deaths in country } c \text{ in month } s)$:

$$\text{conflict}_k(t) = \sum_{c \in \text{top 10 at } t} w_{c,t} \, \big(I_{c,t-1} - I_{c,t-1-k}\big)$$

It measures **how much the conflicts got worse** over the last $k$ known months, in the countries people came from recently. The weights stay fixed inside the formula, so a change in the Swiss nationality mix does not move the index. We decided on this formula and on $k = 3$ and $k = 6$ before running any model.

In [12]:
def conflict_change(t, k):
    w = shares.iloc[t]
    if w.isna().all():
        return np.nan
    top = w.nlargest(10)
    m1 = months[t] - pd.DateOffset(months=1)
    m0 = months[t] - pd.DateOffset(months=1 + k)
    if m1 not in intensity.index or m0 not in intensity.index:
        return np.nan
    change = intensity.loc[m1, top.index] - intensity.loc[m0, top.index]
    weighted = top * change
    return float(weighted.sum()) if weighted.notna().all() else np.nan    # NaN if a month is not in UCDP yet

conflict_3 = np.array([conflict_change(t, 3) for t in range(len(months))])
conflict_6 = np.array([conflict_change(t, 6) for t in range(len(months))])
print("Conflict index available from", months[np.argmax(~np.isnan(conflict_3))].strftime("%Y-%m"))
pd.DataFrame({"conflict_3_months": conflict_3, "conflict_6_months": conflict_6}, index=months).loc[
    ["2015-06-01", "2021-08-01", "2021-09-01", "2021-10-01", "2023-11-01"]].round(3)

Conflict index available from 2008-12


,conflict_3_months,conflict_6_months
month,,
2015-06-01,0.001,0.036
2021-08-01,0.080,0.002
2021-09-01,-0.006,0.676
2021-10-01,-0.677,-0.285
2023-11-01,0.449,0.211


**What we see.** The index exists from December 2008 (the first origin with 12 months of nationality data). It reacts to known events: after the fall of Kabul, the 6-month escalation jumps to 0.68 at the origin of September 2021 (August deaths against February), when Afghans were 19% of the Swiss applications. In October 2021 the 3-month index falls to −0.68, because the fighting stopped. In normal months the index is close to 0.

## B3. The blocks and their combinations

Each block adds two features, ending at the newest month known at the cutoff (`01b`, sections 4 and B6):

| Block | Features | Newest month used |
|---|---|---|
| Neighbours (Eurostat) | change of the log applications in DE + AT + FR + IT: last month, and over 3 months | $t-1$ |
| Mediterranean routes (Frontex numbers at the UNHCR timing) | change of the log arrivals: last month, and over 3 months | $t$ |
| Conflict index (UCDP, weighted by nationality) | change over 3 and over 6 months | $t-1$ |

The Frontex routes of Part A are left out: they are almost the same numbers as the Mediterranean routes, one month later (see above).

In [13]:
features = {
    "neighbours": {
        "neighbours_change_last_month": lambda t: log_neighbours[t - 1] - log_neighbours[t - 2],
        "neighbours_change_3_months": lambda t: log_neighbours[t - 1] - log_neighbours[t - 4],
    },
    "Med routes": {
        "med_change_last_month": lambda t: log_med[t] - log_med[t - 1],
        "med_change_3_months": lambda t: log_med[t] - log_med[t - 3],
    },
    "conflict": {
        "conflict_3_months": lambda t: conflict_3[t],
        "conflict_6_months": lambda t: conflict_6[t],
    },
}
indicators = list(features)                      # neighbours, Med routes, conflict (the Frontex routes: Part A only)
combinations = {"neighbours": ["neighbours"], "Med routes": ["Med routes"], "conflict": ["conflict"],
                "all indicators": indicators}
for removed in indicators:
    combinations[f"all but {removed}"] = [b for b in indicators if b != removed]
# blocks: the features of each combination
blocks = {name: {k: f for b in parts for k, f in features[b].items()} for name, parts in combinations.items()}

def make_table(h, block):
    # Same table as in 02b, plus the features of one block (never later than the month known at the cutoff)
    extra_features = blocks[block]
    rows = []
    for t in range(12, len(log_y) - h):
        values = {name: f(t) for name, f in extra_features.items()}
        if any(np.isnan(v) for v in values.values()):     # a block starts when all its months exist
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            **values,
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    features = ["change_last_month", "change_month_before", "same_change_last_year"] + list(extra_features)
    X = pd.concat([table[features], dummies], axis=1)
    return X, table["target"]

tables = {(block, h): make_table(h, block) for block in blocks for h in range(1, H_MAX + 1)}
for block in blocks:
    X2, _ = tables[(block, 2)]
    print(f"{block:11s} h = 2: {X2.shape[0]} rows, {X2.shape[1]} features, first origin {months[X2.index.min()]:%Y-%m}, "
          f"missing values {X2.isna().sum().sum()}")

neighbours  h = 2: 211 rows, 16 features, first origin 2009-01, missing values 0
Med routes  h = 2: 208 rows, 16 features, first origin 2009-04, missing values 0
conflict    h = 2: 205 rows, 16 features, first origin 2009-01, missing values 0
all indicators h = 2: 202 rows, 20 features, first origin 2009-04, missing values 0
all but neighbours h = 2: 202 rows, 18 features, first origin 2009-04, missing values 0
all but Med routes h = 2: 205 rows, 18 features, first origin 2009-01, missing values 0
all but conflict h = 2: 208 rows, 18 features, first origin 2009-04, missing values 0


**What we see.** For $h = 2$, each single block gives 16 features (the three own changes, two indicator features, 11 month dummies), the model with all indicators 20, and each "all but one" model 18. No value is missing. The tables start in January 2009, or in April 2009 when the Mediterranean routes are in (they need the arrivals back to $t-3$), so the models have up to 3 fewer training rows than `02b`. (The tables also contain rows after the backtest, which are never used here; the conflict index has none after January 2026, because UCDP ends in December 2025.)

## B4. Models, grids and the choice of `alpha`: the same as in `02b`

Copied from Part A without any change (only the model names lose the "+ routes").

In [14]:
def make_model(name, alpha=None):
    if name == "Linear regression":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression": [None],
    "Ridge": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def choose_alpha(name, X, target, T, h, n_blocks=2, block=12):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    first_row, last_row = X.index.min(), T - h
    scores = []
    for alpha in grid:
        mse_blocks = []
        for b in range(n_blocks):
            start = last_row - n_blocks * block + 1 + b * block
            valid_rows = list(range(start, start + block))
            train_rows = list(range(first_row, start - h + 1))
            model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
            mse_blocks.append(np.mean(errors ** 2))
        scores.append(np.mean(mse_blocks))
    return grid[int(np.argmin(scores))]

## B5. The backtest loop

The same loop as in `02b` and Part A, once per combination: the same 141 origins and horizons, and the same check that no training target lies after the origin (`assert`). This takes about eight minutes.

In [15]:
rows = []
for block in blocks:
    for t in range(first_origin, last_backtest):
        for h in range(1, H_MAX + 1):
            target_pos = t + h
            if target_pos > last_backtest:
                continue
            X, target = tables[(block, h)]
            train_rows = list(range(X.index.min(), t - h + 1))
            assert max(train_rows) + h <= t                    # no training target after the origin

            for name in alpha_grids:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    alpha = choose_alpha(name, X, target, t, h)
                    model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
                forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
                rows.append({
                    "model": f"{name} + {block}", "block": block, "origin": months[t], "month": months[target_pos],
                    "h": h, "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                    "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha,
                })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 8820


## B6. Checks on the stored forecasts

In [16]:
print(forecasts.groupby(["block", "h"]).size().unstack())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

h                     1    2    3
block                            
Med routes          423  420  417
all but Med routes  423  420  417
all but conflict    423  420  417
all but neighbours  423  420  417
all indicators      423  420  417
conflict            423  420  417
neighbours          423  420  417

All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True
Missing or infinite forecasts: 0


**What we see.** As in `02b` and Part A: for each of the seven combinations, 141, 140 and 139 forecasts per model for $h = 1, 2, 3$ (423, 420 and 417 for the three models), 8,820 together, all in the backtest, all after their origin, none missing. With the `assert` in the loop, this shows that the indicators used only months known at each origin.

### Which penalties were chosen?

As in Part A (notebook 209), we check that the chosen `alpha` does not sit mostly at an edge of the grid.

In [17]:
for name in ["Ridge", "Lasso"]:
    chosen = forecasts[forecasts["model"].str.startswith(name + " +")]
    print(f"{name}: how often each alpha was chosen, per block")
    print(chosen.groupby(["block", "alpha"]).size().unstack(fill_value=0).to_string())
    print()

Ridge: how often each alpha was chosen, per block
alpha               0.1      1.0      10.0     100.0    1000.0   10000.0
block                                                                   
Med routes               55        9       66      269       20        1
all but Med routes       48       17       81      182       58       34
all but conflict         27       21       76      264       30        2
all but neighbours       64        4       83      248       20        1
all indicators           24        5       85      271       33        2
conflict                 85        6       45      217       61        6
neighbours               63       25       72      174       56       30

Lasso: how often each alpha was chosen, per block
alpha               0.0001  0.0010  0.0030  0.0100  0.0300  0.1000  0.3000
block                                                                     
Med routes              57      33      16     165      70      30      49
all but Med route

**What we see.** Ridge most often chose `alpha = 100` in every combination (174 to 271 of 420 times); the largest value (10,000) was chosen at most 34 times. Lasso most often chose 0.01 or 0.03; the two edges of its grid, 0.0001 and 0.3, were chosen up to 83 and 50 times (about 20% and 12% of the forecasts).

**What this means for us.** As in Part A, the chosen values lie mostly inside the grids, so the grids are wide enough.

## B7. What do the models do with the extra data?

As in Part A (section 6), the coefficients for $h = 2$ at the last origin (August 2024), on standardised features (notebook 105), for the model with all indicators. Lasso can set a coefficient to exactly 0 (notebook 105), so it shows which sources it keeps. A coefficient says what helps to *predict*, not what *causes*.

In [18]:
t = last_backtest - 1
h = 2
X, target = tables[("all indicators", h)]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in ["Linear regression", "Ridge", "Lasso"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_

print("Origin:", months[t], "| horizon h =", h, "| training rows:", len(train_rows))
coefficients.iloc[:11].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | training rows: 183


,Linear regression (alpha = None),Ridge (alpha = 10),Lasso (alpha = 0.003)
change_last_month,-0.039,-0.035,-0.031
change_month_before,-0.018,-0.015,-0.008
same_change_last_year,0.017,0.019,0.018
neighbours_change_last_month,0.014,0.007,0.000
neighbours_change_3_months,-0.062,-0.055,-0.052
med_change_last_month,0.051,0.047,0.047
med_change_3_months,0.035,0.035,0.034
conflict_3_months,0.022,0.019,0.014
conflict_6_months,-0.026,-0.022,-0.015
month_2,-0.015,-0.014,-0.010


Which indicators does Lasso keep? At every origin of the backtest ($h = 2$, all indicators), we record which coefficients Lasso sets to exactly 0 (notebook 105). A feature that is kept at most origins carries information that the series' own past does not.

In [19]:
extra_names = list(blocks["all indicators"])
kept = []
for t in range(first_origin, last_backtest - 1):          # all origins with a known h = 2 target
    X, target = tables[("all indicators", 2)]
    train_rows = list(range(X.index.min(), t - 2 + 1))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha("Lasso", X, target, t, 2)
        model = make_model("Lasso", alpha).fit(X.loc[train_rows], target.loc[train_rows])
    kept.append(pd.Series(model[-1].coef_, index=X.columns)[extra_names])
kept = pd.DataFrame(kept)
lasso_kept = pd.DataFrame({"block": [next(b for b in indicators if k in features[b]) for k in extra_names],
                           "share of origins kept": (kept != 0).mean().round(2),
                           "mean coefficient": kept.mean().round(3)})
lasso_kept.index.name = "feature"
lasso_kept.to_csv("../output/tables/02c_lasso_kept.csv")
print("Origins:", len(kept))
lasso_kept

Origins: 140


,block,share of origins kept,mean coefficient
feature,,,
neighbours_change_last_month,neighbours,0.44,0.005
neighbours_change_3_months,neighbours,0.60,-0.022
med_change_last_month,Med routes,0.81,0.024
med_change_3_months,Med routes,0.96,0.041
conflict_3_months,conflict,0.25,0.005
conflict_6_months,conflict,0.39,-0.005


**What we see.**
- At the last origin (183 training rows), the three models agree. The largest indicator coefficients are the two **Mediterranean route** changes (+0.047 and +0.034 for Lasso) and the 3-month change of the **neighbours** (−0.052). The conflict features are smaller (+0.014 and −0.015). The two neighbour features have opposite signs, as the two route features had in Part A, so neither sign should be read on its own.
- Over all 140 origins, Lasso keeps the 3-month change of the Mediterranean routes at **96%** of the origins and its monthly change at 81%. The neighbours are kept at 44% and 60%, the conflict features at only 25% and 39%.

**What this means for us.** If one indicator carries information that the series' own past does not, it is the Mediterranean routes at the UNHCR timing; the conflict index carries the least. Whether this lowers the *forecast errors* is measured in `03_evaluate.ipynb`, Part B (sections 12 and 13).

## B8. Save the forecasts

In [20]:
forecasts.to_csv("../data/processed/backtest_forecasts_extra.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 8820 forecasts with the columns ['model', 'block', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha']


## Summary

**Part A: the route indicator**

- **Models:** the three regressions of `02b` (linear regression, Ridge, Lasso), with the same features, target, grids and validation, plus two route features: the change of log detections in the last known month and over the last three known months.
- **Timing rule (from `01b`):** at origin $t$ we use the indicator only up to month $t - 1$, because Frontex is published one month later than the target.
- **Rules as before:** the same 141 origins and horizons; scaling and the choice of `alpha` happen inside each training window; no training target after the origin (checked with `assert`); the test period is not used.
- **Not seen in class (marked above):** `reindex(months)`.
- **Output:** 1,260 forecasts in `../data/processed/backtest_forecasts_indicator.csv`, compared in `03_evaluate.ipynb` with the same models without the indicator.

**Part B: all indicators (added after the test)**

- **Models:** the three regressions of `02b` with indicators: each block alone (neighbours, Mediterranean routes, conflict index), all three together ("all indicators", 6 extra features), and all indicators minus one block, three times. Same target, features, grids, validation and 141 origins as `02b`.
- **One route block only:** the Frontex routes of Part A are left out, because the Mediterranean routes carry the same numbers one month earlier and did better in the backtest.
- **Checks:** 8,820 forecasts, all in the backtest, after their origin, none missing; the chosen `alpha` lies mostly inside the grids.
- **Lasso with all indicators** keeps the 3-month change of the Mediterranean routes at 96% of the origins, the conflict features at only 25% to 39%.
- **Output:** `../data/processed/backtest_forecasts_extra.csv` and `../output/tables/02c_lasso_kept.csv`, evaluated in `03_evaluate.ipynb`, Part B.